In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import mhnlib.utils as mhn_utils
from math import log, sqrt
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from typing import Optional, Tuple, Union
from pathlib import Path
import seaborn as sns
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

# IID patterns

## Generate data (Quenching/Annealing)

In [ ]:
torch.manual_seed(1101252)
N = 16
K_max = 256
Ks = torch.arange(16,K_max+1, step=8)
patterns_per_K = {}
for K in Ks:
    S = max(3*K, 128)
    patterns_per_K[K.item()] = torch.randn(S, K.item(), N)/sqrt(N)

### Quenching

In [ ]:
overwrite = True
torch.manual_seed(1101252)
num_betas = 100
num_iterations = 10000
num_ics = 5*N
ics = torch.randn(num_ics, N)/sqrt(N)
betas_quench = torch.logspace(-1, 2, steps=num_betas)
for K, patterns in patterns_per_K.items():
    print(f'Processing K={K}...')
    #grams = mhn_utils.get_gram_matrix(patterns)
    #symm_stability_matrix_unif = mhn_utils.get_symmetric_stability_matrix(grams, torch.ones(K)/K)
    #stability_unif_vals = torch.linalg.eigvalsh(symm_stability_matrix_unif)
    #beta_c_unif = 1.0/stability_unif_vals[:,-1]
    #reference_log_beta = int(beta_c_unif.min().log10())
    
    
    for is_centered in [False, True]:
        filename = f'paper_results/local_data/iid_centered={is_centered}_quenched_N={N}_K={K}.npz'
        if Path(filename).exists() and not overwrite:
            print(f'File {filename} already exists, skipping...')
            continue
        if device.type == 'mps':
            torch.mps.empty_cache()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        # output shape is (num_betas, num_pattern_samples, num_ics, N)
        # and (num_betas, num_pattern_samples, num_ics, K)
        x_quench_fps, probs_quench_fps = mhn_utils.deterministic_dynamics((patterns - patterns.mean(dim=-2, keepdim=True)).to(device)  if is_centered else patterns.to(device),
                                                                          torch.zeros(K).to(device),
                                                                          betas_quench.to(device),
                                                                          ics.to(device),
                                                                          num_iterations,
                                                                          return_probs=True,
                                                                          verbose=True)
        x_quench_fps = x_quench_fps.cpu()
        probs_quench_fps = probs_quench_fps.cpu()
        entropies_quench_fps = mhn_utils.get_entropy(probs_quench_fps)
        np.savez(filename,
                 patterns=patterns_per_K[K].numpy(),
                 x=x_quench_fps.numpy(),
                 probs=probs_quench_fps.numpy(),
                 entropies=entropies_quench_fps.numpy(),
                 betas=betas_quench.numpy())

### Annealing

In [ ]:
torch.manual_seed(1101252)
overwrite = False
num_betas = 100
num_iterations = 10000
num_ics = 3*N
logit_noise_std = 0.1
ics = torch.randn(num_ics, N)/sqrt(N)
betas_anneal = torch.logspace(-1, 2, steps=num_betas)
for K, patterns in patterns_per_K.items():
    print(f'Processing K={K}...')
    #grams = mhn_utils.get_gram_matrix(patterns)
    #symm_stability_matrix_unif = mhn_utils.get_symmetric_stability_matrix(grams, torch.ones(K)/K)
    #stability_unif_vals = torch.linalg.eigvalsh(symm_stability_matrix_unif)
    #beta_c_unif = 1.0/stability_unif_vals[:,-1]
    #reference_log_beta = int(beta_c_unif.min().log10())
    
    for is_centered in [False, True]:
        filename = f'paper_results/local_data/iid_centered={is_centered}_annealed_N={N}_K={K}.npz'
        if Path(filename).exists() and not overwrite:
            print(f'File {filename} already exists, skipping...')
            continue
        if device.type == 'mps':
            torch.mps.empty_cache()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        # output shape is (num_betas, num_pattern_samples, num_ics, N)
        # and (num_betas, num_pattern_samples, num_ics, K)
        
        x_anneal_fps, probs_anneal_fps = mhn_utils.deterministic_dynamics_annealing((patterns - patterns.mean(dim=-2, keepdim=True)).to(device)  if is_centered else patterns.to(device),
                                                                                  torch.zeros(K).to(device),
                                                                                  betas_anneal.to(device),
                                                                                  ics.to(device),
                                                                                  logit_noise_std=logit_noise_std,
                                                                                  num_iterations=num_iterations,
                                                                                  return_probs=True,
                                                                                  verbose=True)

        x_anneal_fps = x_anneal_fps.cpu()
        probs_anneal_fps = probs_anneal_fps.cpu()
        entropies_anneal_fps = mhn_utils.get_entropy(probs_anneal_fps)
        np.savez(filename,
                 patterns=patterns_per_K[K].numpy(),
                 x=x_anneal_fps.numpy(),
                 probs=probs_anneal_fps.numpy(),
                 entropies=entropies_anneal_fps.numpy(),
                 betas=betas_anneal.numpy())

## Generate data (Dual Quenching/Annealing)

In [ ]:
torch.manual_seed(1101252)
N = 16
K_max = 256
Ks = torch.arange(16,K_max+1, step=8)
patterns_per_K = {}
for K in Ks:
    S = max(2*K, 128)
    patterns_per_K[K.item()] = torch.randn(S, K.item(), N)/sqrt(N)

### Quenching (dual)

In [ ]:
overwrite = False
torch.manual_seed(1101252)
num_betas = 100
num_iterations = 10000
betas_quench = torch.logspace(-1, 2, steps=num_betas)
for K, patterns in patterns_per_K.items():
    print(f'Processing K={K}...')
    #grams = mhn_utils.get_gram_matrix(patterns)
    #symm_stability_matrix_unif = mhn_utils.get_symmetric_stability_matrix(grams, torch.ones(K)/K)
    #stability_unif_vals = torch.linalg.eigvalsh(symm_stability_matrix_unif)
    #beta_c_unif = 1.0/stability_unif_vals[:,-1]
    #reference_log_beta = int(beta_c_unif.min().log10())
    weights_uniform = torch.ones(K)/K
    for is_centered in [False, True]:
        filename = f'paper_results/local_data/iid_weights_centered={is_centered}_quenched_N={N}_K={K}.npz'
        if Path(filename).exists() and not overwrite:
            print(f'File {filename} already exists, skipping...')
            continue
        if device.type == 'mps':
            torch.mps.empty_cache()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        grams = mhn_utils.get_gram_matrix(patterns - patterns.mean(dim=-2, keepdim=True)) if is_centered else  mhn_utils.get_gram_matrix(patterns)
        w_quench_fps = mhn_utils.dual_deterministic_dynamics(grams.to(device),
                                                            torch.zeros(K).to(device),
                                                            betas_quench.to(device),
                                                            weights_uniform.to(device),
                                                            num_iterations,
                                                            verbose=True)
        w_quench_fps = w_quench_fps.cpu().squeeze()
        entropies_quench_fps = mhn_utils.get_entropy(w_quench_fps)
        np.savez(filename,
                 patterns=patterns_per_K[K].numpy(),
                 probs=w_quench_fps.numpy(),
                 entropies=entropies_quench_fps.numpy(),
                 betas=betas_quench.numpy())

### Annealing (dual)

In [ ]:
torch.manual_seed(1101252)
overwrite = True
num_betas = 100
num_iterations = 10000
num_ics = 3*N
logit_noise_std = 0.1
betas_anneal = torch.logspace(-1, 2, steps=num_betas)
for K, patterns in patterns_per_K.items():
    print(f'Processing K={K}...')
    weights_ic = torch.softmax(torch.randn(num_ics, K)*logit_noise_std, dim=-1)

    for is_centered in [False, True]:
        filename = f'paper_results/local_data/iid_weights_centered={is_centered}_annealed_N={N}_K={K}.npz'
        if Path(filename).exists() and not overwrite:
            print(f'File {filename} already exists, skipping...')
            continue
        if device.type == 'mps':
            torch.mps.empty_cache()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            
        grams = mhn_utils.get_gram_matrix(patterns - patterns.mean(dim=-2, keepdim=True)) if is_centered else  mhn_utils.get_gram_matrix(patterns)
        w_anneal_fps = mhn_utils.dual_deterministic_dynamics_annealing(grams.to(device),
                                                                    torch.zeros(K).to(device),
                                                                    betas_anneal.to(device),
                                                                    weights_ic.to(device),
                                                                    logit_noise_std=logit_noise_std,
                                                                    num_iterations=num_iterations,
                                                                    verbose=True)
        w_anneal_fps = w_anneal_fps.cpu().squeeze()
        entropies_anneal_fps = mhn_utils.get_entropy(w_anneal_fps)
        np.savez(filename,
                 patterns=patterns_per_K[K].numpy(),
                 probs=w_anneal_fps.numpy(),
                 entropies=entropies_anneal_fps.numpy(),
                 betas=betas_anneal.numpy())
        

## Load generated data

In [ ]:
is_centered = True
data_folder = Path('paper_results/local_data/')
N = 16
#patterns_per_K = {}
betas_quenched_per_K = {}
betas_annealed_per_K = {}
#x_quenched_fps_per_K = {}
#probs_quenched_fps_per_K = {}
entropies_quenched_per_K = {}
#x_annealed_fps_per_K = {}
#probs_annealed_fps_per_K = {}
entropies_annealed_per_K = {}
for quenched_file in tqdm(list(data_folder.glob(f'iid_centered={is_centered}_quenched_N={N}_K=*.npz')), desc='Loading data...'):
    data = np.load(quenched_file, allow_pickle=True)
    K = data['patterns'].shape[1]
    #patterns_per_K[K] = data['patterns']
    betas_quenched_per_K[K] = torch.as_tensor(data['betas'])
    #x_quenched_fps_per_K[K] = torch.as_tensor(data['x'])
    #probs_quenched_fps_per_K[K] = torch.as_tensor(data['probs'])
    entropies_quenched_per_K[K] = torch.as_tensor(data['entropies'])
    annealed_file = quenched_file.parent / f'iid_centered={is_centered}_annealed_N={N}_K={K}.npz'
    if annealed_file.exists():
        data = np.load(annealed_file, allow_pickle=True)
        betas_annealed_per_K[K] = torch.as_tensor(data['betas'])
        #x_annealed_fps_per_K[K] = torch.as_tensor(data['x'])
        #probs_annealed_fps_per_K[K] = torch.as_tensor(data['probs'])
        entropies_annealed_per_K[K] = torch.as_tensor(data['entropies'])


## Plots

In [ ]:
all_Ks = list(sorted(entropies_quenched_per_K.keys()))
mean_entropies_quenched = []
mean_entropies_annealed = []
for K in all_Ks:
    mean_entropies_quenched.append(entropies_quenched_per_K[K].mean(dim=(1,2)))
    mean_entropies_annealed.append(entropies_annealed_per_K[K].mean(dim=(1,2)))
    betas = betas_quenched_per_K[K]
mean_entropies_quenched = torch.stack(mean_entropies_quenched)
mean_entropies_annealed = torch.stack(mean_entropies_annealed)
all_Ks = torch.tensor(all_Ks)

In [ ]:
beta_c_analytical = all_Ks/(1+all_Ks/N + 2*torch.sqrt(all_Ks/N))



fig, ax = plt.subplots(figsize=(4, 3))
ctf = ax.contourf(betas, all_Ks, mean_entropies_quenched/torch.log(all_Ks)[:,None], levels=100, cmap='coolwarm', vmin=0.0, vmax=1.0)
plt.colorbar(ctf, label='$H(w)/log(K)$')
plt.plot(beta_c_analytical.cpu(), all_Ks.cpu(), 'k--', label='$\\beta_c$ analytical')
ax.set_xscale('log')
ax.set_xlabel('Beta')
ax.set_ylabel('K')
ax.set_title('Quenched Entropies')
plt.tight_layout()
plt.savefig(f'paper_results/plots/iid_centered={is_centered}_quenched_N={N}_entropies.pdf', bbox_inches='tight')
plt.show()


fig, ax = plt.subplots(figsize=(4, 3))
ctf = ax.contourf(betas, all_Ks, mean_entropies_annealed/torch.log(all_Ks)[:,None], levels=100, cmap='coolwarm', vmin=0.0, vmax=1.0)
plt.colorbar(ctf, label='$H(w)/log(K)$')
plt.plot(beta_c_analytical.cpu(), all_Ks.cpu(), 'k--', label='$\\beta_c$ analytical')
ax.set_xscale('log')
ax.set_xlabel('Beta')
ax.set_ylabel('K')
ax.set_title('Annealed Entropies')
plt.tight_layout()
plt.savefig(f'paper_results/plots/iid_centered={is_centered}_annealed_N={N}_entropies.pdf', bbox_inches='tight')
plt.show()

## Load generated data (dual)

In [ ]:
is_centered = True
data_folder = Path('paper_results/local_data/')
N = 16
betas_dual_quenched_per_K = {}
betas_dual_annealed_per_K = {}
entropies_dual_quenched_per_K = {}
entropies_dual_annealed_per_K = {}
for quenched_file in tqdm(list(data_folder.glob(f'iid_weights_centered={is_centered}_quenched_N={N}_K=*.npz')), desc='Loading data...'):
    data = np.load(quenched_file, allow_pickle=True)
    K = data['patterns'].shape[1]
    betas_dual_quenched_per_K[K] = torch.as_tensor(data['betas'])
    entropies_dual_quenched_per_K[K] = torch.as_tensor(data['entropies'])
    annealed_file = quenched_file.parent / f'iid_weights_centered={is_centered}_annealed_N={N}_K={K}.npz'
    if annealed_file.exists():
        data = np.load(annealed_file, allow_pickle=True)
        betas_dual_annealed_per_K[K] = torch.as_tensor(data['betas'])
        entropies_dual_annealed_per_K[K] = torch.as_tensor(data['entropies'])

## Plots (dual)

In [ ]:
all_Ks_quenched = list(sorted(betas_dual_quenched_per_K.keys()))
all_Ks_annealed = list(sorted(betas_dual_annealed_per_K.keys()))
mean_dual_entropies_quenched = []
mean_dual_entropies_annealed = []
for K in all_Ks_quenched:
    mean_dual_entropies_quenched.append(entropies_dual_quenched_per_K[K].mean(dim=1))
    betas_quenched = betas_dual_quenched_per_K[K]
for K in all_Ks_annealed:
    mean_dual_entropies_annealed.append(entropies_dual_annealed_per_K[K].mean(dim=(1,2)))
    betas_annealed = betas_dual_annealed_per_K[K]
mean_dual_entropies_quenched = torch.stack(mean_dual_entropies_quenched)
mean_dual_entropies_annealed = torch.stack(mean_dual_entropies_annealed)
all_Ks_quenched = torch.tensor(all_Ks_quenched)
all_Ks_annealed = torch.tensor(all_Ks_annealed)

In [ ]:
beta_c_analytical_quenched = all_Ks_quenched/(1+all_Ks_quenched/N + 2*torch.sqrt(all_Ks_quenched/N))
beta_c_analytical_annealed = all_Ks_annealed/(1+all_Ks_annealed/N + 2*torch.sqrt(all_Ks_annealed/N))
fig, ax = plt.subplots(figsize=(4, 3))
ctf = ax.contourf(betas, all_Ks_quenched, mean_dual_entropies_quenched/torch.log(all_Ks_quenched)[:,None], levels=100, cmap='coolwarm', vmin=0.0, vmax=1.0)
plt.colorbar(ctf, label='$H(w)/log(K)$')
plt.plot(beta_c_analytical_quenched.cpu(), all_Ks_quenched.cpu(), 'k--', label='$\\beta_c$ analytical')
ax.set_xscale('log')
ax.set_xlabel('Beta')
ax.set_ylabel('K')
ax.set_title('Quenched Entropies')
plt.tight_layout()
plt.savefig(f'paper_results/plots/iid_weights_centered={is_centered}_quenched_N={N}_entropies.pdf', bbox_inches='tight')
plt.show()


fig, ax = plt.subplots(figsize=(4, 3))
ctf = ax.contourf(betas, all_Ks_annealed, mean_dual_entropies_annealed/torch.log(all_Ks_annealed)[:,None], levels=100, cmap='coolwarm', vmin=0.0, vmax=1.0)
plt.colorbar(ctf, label='$H(w)/log(K)$')
plt.plot(beta_c_analytical_annealed.cpu(), all_Ks_annealed.cpu(), 'k--', label='$\\beta_c$ analytical')
ax.set_xscale('log')
ax.set_xlabel('Beta')
ax.set_ylabel('K')
ax.set_title('Annealed Entropies')
plt.tight_layout()
plt.savefig(f'paper_results/plots/iid_weights_centered={is_centered}_annealed_N={N}_entropies.pdf', bbox_inches='tight')
plt.show()

# 1-step hierarchical matrices, vary $M$, fixed $\rho_0$ and $\rho_1$

## Generate data

In [ ]:
torch.manual_seed(1101252)
K = 256
N = 32
S = 100
Ms = 2**torch.arange(0, torch.log2(torch.tensor(S)).int()+1)
rho0 = 0.1
rho1 = 0.8
patterns_per_M = {}
for M in Ms:
    global_patterns = sqrt(rho0)*torch.randn(S, N)/sqrt(N)
    block_patterns = sqrt(rho1-rho0)*torch.randn(S, M, N)/sqrt(N)
    local_patterns = sqrt(1-rho1)*torch.randn(S,K, N)/sqrt(N)
    patterns = global_patterns[:,None, :] + torch.repeat_interleave(block_patterns, K//M, dim=1) + local_patterns
    patterns_per_M[M.item()] = patterns

### Quenching

In [ ]:
overwrite = True
torch.manual_seed(1101252)
num_betas = 200
num_iterations = 10000
num_ics = 3*N
ics = torch.randn(num_ics, N)/sqrt(N)
betas_quench = torch.logspace(-1, 2, steps=num_betas)
for M, patterns in patterns_per_M.items():
    print(f'Processing M={M}...')
    for is_centered in [False, True]:
        filename = f'paper_results/local_data/onestep_centered={is_centered}_quenched_N={N}_K={K}_M={M}_rho0={rho0:.2f}_rho1={rho1:.2f}.npz'
        if Path(filename).exists() and not overwrite:
            print(f'File {filename} already exists, skipping...')
            continue
        if device.type == 'mps':
            torch.mps.empty_cache()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        # output shape is (num_betas, num_pattern_samples, num_ics, N)
        # and (num_betas, num_pattern_samples, num_ics, K)
        x_quench_fps, probs_quench_fps = mhn_utils.deterministic_dynamics((patterns - patterns.mean(dim=-2, keepdim=True)).to(device)  if is_centered else patterns.to(device),
                                                                          torch.zeros(K).to(device),
                                                                          betas_quench.to(device),
                                                                          ics.to(device),
                                                                          num_iterations,
                                                                          return_probs=True,
                                                                          verbose=True)
        x_quench_fps = x_quench_fps.cpu()
        probs_quench_fps = probs_quench_fps.cpu()
        entropies_quench_fps = mhn_utils.get_entropy(probs_quench_fps)
        np.savez(filename,
                 patterns=patterns_per_M[M].numpy(),
                 x=x_quench_fps.numpy(),
                 probs=probs_quench_fps.numpy(),
                 entropies=entropies_quench_fps.numpy(),
                 betas=betas_quench.numpy())

### Annealing

In [ ]:
torch.manual_seed(1101252)
overwrite = True
num_betas = 200
num_iterations = 10000
num_ics = 3*N
logit_noise_std = 0.1
ics = torch.randn(num_ics, N)/sqrt(N)
betas_anneal = torch.logspace(-1, 2, steps=num_betas)
for M, patterns in patterns_per_M.items():
    print(f'Processing M={M}...')
    for is_centered in [False, True]:
        filename = f'paper_results/local_data/onestep_centered={is_centered}_annealed_N={N}_K={K}_M={M}_rho0={rho0:.2f}_rho1={rho1:.2f}.npz'
        if Path(filename).exists() and not overwrite:
            print(f'File {filename} already exists, skipping...')
            continue
        if device.type == 'mps':
            torch.mps.empty_cache()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        # output shape is (num_betas, num_pattern_samples, num_ics, N)
        # and (num_betas, num_pattern_samples, num_ics, K)
        
        x_anneal_fps, probs_anneal_fps = mhn_utils.deterministic_dynamics_annealing((patterns - patterns.mean(dim=-2, keepdim=True)).to(device)  if is_centered else patterns.to(device),
                                                                                  torch.zeros(K).to(device),
                                                                                  betas_anneal.to(device),
                                                                                  ics.to(device),
                                                                                  logit_noise_std=logit_noise_std,
                                                                                  num_iterations=num_iterations,
                                                                                  return_probs=True,
                                                                                  verbose=True)

        x_anneal_fps = x_anneal_fps.cpu()
        probs_anneal_fps = probs_anneal_fps.cpu()
        entropies_anneal_fps = mhn_utils.get_entropy(probs_anneal_fps)
        np.savez(filename,
                 patterns=patterns_per_M[M].numpy(),
                 x=x_anneal_fps.numpy(),
                 probs=probs_anneal_fps.numpy(),
                 entropies=entropies_anneal_fps.numpy(),
                 betas=betas_anneal.numpy())

## Generate data (dual)

In [ ]:
torch.manual_seed(1101252)
K = 256
N = 32
S = 2*K
Ms = 2**torch.arange(0, torch.log2(torch.tensor(K)).int()+1)
rho0 = 0.1
rho1 = 0.9
patterns_per_M = {}
for M in Ms:
    global_patterns = sqrt(rho0)*torch.randn(S, N)/sqrt(N)
    block_patterns = sqrt(rho1-rho0)*torch.randn(S, M, N)/sqrt(N)
    local_patterns = sqrt(1-rho1)*torch.randn(S,K, N)/sqrt(N)
    patterns = global_patterns[:,None, :] + torch.repeat_interleave(block_patterns, K//M, dim=1) + local_patterns
    patterns_per_M[M.item()] = patterns

### Quenching (dual)

In [ ]:
overwrite = True
torch.manual_seed(1101252)
num_betas = 100
num_iterations = 10000
num_ics_per_batch = 3
num_batches = N
num_ics = num_batches*num_ics_per_batch
logit_noise_std = 1/sqrt(N)
betas_quench = torch.logspace(-1, 2, steps=num_betas)
for M, patterns in patterns_per_M.items():
    print(f'Processing M={M}...')
    #grams = mhn_utils.get_gram_matrix(patterns)
    #symm_stability_matrix_unif = mhn_utils.get_symmetric_stability_matrix(grams, torch.ones(K)/K)
    #stability_unif_vals = torch.linalg.eigvalsh(symm_stability_matrix_unif)
    #beta_c_unif = 1.0/stability_unif_vals[:,-1]
    #reference_log_beta = int(beta_c_unif.min().log10())
    #weights_uniform = torch.ones(K)/K
    weights_ic = torch.softmax(torch.randn(num_ics, K)*logit_noise_std, dim=-1)
    for is_centered in [False, True]:
        
        filename = f'paper_results/local_data/onestep_weights_centered={is_centered}_quenched_N={N}_K={K}_M={M}_rho0={rho0:.2f}_rho1={rho1:.2f}.npz'
        if Path(filename).exists() and not overwrite:
            print(f'File {filename} already exists, skipping...')
            continue
        if device.type == 'mps':
            torch.mps.empty_cache()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        grams = mhn_utils.get_gram_matrix(patterns - patterns.mean(dim=-2, keepdim=True)) if is_centered else  mhn_utils.get_gram_matrix(patterns)
        grams = grams.to(device)
        biases = torch.zeros(K).to(device)
        w_quench_fps = []
        for batch_idx in range(num_batches):
            batch_start = batch_idx*num_ics_per_batch
            batch_end = (batch_idx+1)*num_ics_per_batch
            w_quench_fps_batch = mhn_utils.dual_deterministic_dynamics(grams,
                                                                biases,
                                                                betas_quench.to(device),
                                                                weights_ic[batch_start:batch_end].to(device),
                                                                num_iterations,
                                                                verbose=True)
            w_quench_fps_batch = w_quench_fps_batch.cpu().squeeze()
            w_quench_fps.append(w_quench_fps_batch)
        w_quench_fps = torch.cat(w_quench_fps, dim=-2)
        entropies_quench_fps = mhn_utils.get_entropy(w_quench_fps)
        np.savez(filename,
                 patterns=patterns_per_M[M].numpy(),
                 probs=w_quench_fps.numpy(),
                 entropies=entropies_quench_fps.numpy(),
                 betas=betas_quench.numpy())
        
        #allocated = torch.cuda.memory_allocated(0) / 1024**3
        #reserved = torch.cuda.memory_reserved(0) / 1024**3
        #peak = torch.cuda.max_memory_allocated(0) / 1024**3
        #print(f"Allocated: {allocated:.2f} GiB")
        #print(f"Reserved:  {reserved:.2f} GiB")
        #print(f"Peak:      {peak:.2f} GiB")
        

### Annealing (dual)

In [ ]:
torch.manual_seed(1101252)
overwrite = True
num_betas = 100
num_iterations = 10000
num_ics = 3*N
logit_noise_std = 1/sqrt(N)
betas_anneal = torch.logspace(-1, 2, steps=num_betas)
for M, patterns in patterns_per_M.items():
    print(f'Processing M={M}...')
    weights_ic = torch.softmax(torch.randn(num_ics, K)*logit_noise_std, dim=-1)

    for is_centered in [False, True]:
        filename = f'paper_results/local_data/onestep_weights_centered={is_centered}_annealed_N={N}_K={K}_M={M}_rho0={rho0:.2f}_rho1={rho1:.2f}.npz'
        if Path(filename).exists() and not overwrite:
            print(f'File {filename} already exists, skipping...')
            continue
        if device.type == 'mps':
            torch.mps.empty_cache()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            
        grams = mhn_utils.get_gram_matrix(patterns - patterns.mean(dim=-2, keepdim=True)) if is_centered else  mhn_utils.get_gram_matrix(patterns)
        w_anneal_fps = mhn_utils.dual_deterministic_dynamics_annealing(grams.to(device),
                                                                    torch.zeros(K).to(device),
                                                                    betas_anneal.to(device),
                                                                    weights_ic.to(device),
                                                                    logit_noise_std=logit_noise_std,
                                                                    num_iterations=num_iterations,
                                                                    verbose=True)
        w_anneal_fps = w_anneal_fps.cpu().squeeze()
        entropies_anneal_fps = mhn_utils.get_entropy(w_anneal_fps)
        np.savez(filename,
                 patterns=patterns_per_M[M].numpy(),
                 probs=w_anneal_fps.numpy(),
                 entropies=entropies_anneal_fps.numpy(),
                 betas=betas_anneal.numpy())

## Load generated data (dual)

In [ ]:
is_centered = True
load_probs = True
data_folder = Path('paper_results/local_data/')
N = 32
K = 256
betas_dual_quenched_per_M = {}
betas_dual_annealed_per_M = {}
entropies_dual_quenched_per_M = {}
entropies_dual_annealed_per_M  = {}
patterns_quenched_per_M = {}
patterns_annealed_per_M = {}
rho0 = 0.1
rho1 = 0.9
if load_probs:
    probs_dual_quenched_per_M = {}
    probs_dual_annealed_per_M = {}
for quenched_file in tqdm(list(data_folder.glob(f'onestep_weights_centered={is_centered}_quenched_N={N}_K={K}_M=*_rho0={rho0:.2f}_rho1={rho1:.2f}.npz')), desc='Loading data...'):
    data = np.load(quenched_file, allow_pickle=True)
    M = int(quenched_file.stem.split('_M=')[1].split('_rho0')[0])
    betas_dual_quenched_per_M[M] = torch.as_tensor(data['betas'])
    entropies_dual_quenched_per_M[M] = torch.as_tensor(data['entropies'])
    if load_probs:
        probs_dual_quenched_per_M[M] = torch.as_tensor(data['probs'])
        patterns_quenched_per_M[M] = torch.as_tensor(data['patterns'])
    annealed_file = quenched_file.parent / f'onestep_weights_centered={is_centered}_annealed_N={N}_K={K}_M={M}_rho0={rho0:.2f}_rho1={rho1:.2f}.npz'
    if annealed_file.exists():
        data = np.load(annealed_file, allow_pickle=True)
        betas_dual_annealed_per_M[M] = torch.as_tensor(data['betas'])
        entropies_dual_annealed_per_M[M] = torch.as_tensor(data['entropies'])
        if load_probs:
            probs_dual_annealed_per_M[M] = torch.as_tensor(data['probs'])
            patterns_annealed_per_M[M] = torch.as_tensor(data['patterns'])

### Plots

In [ ]:
all_Ms = list(sorted(entropies_dual_quenched_per_M.keys()))
mean_dual_entropies_quenched = []
mean_dual_entropies_annealed = []
for M in all_Ms:
    mean_dual_entropies_quenched.append(entropies_dual_quenched_per_M[M].mean(dim=(-1,-2)))
    mean_dual_entropies_annealed.append(entropies_dual_annealed_per_M[M].mean(dim=(-1,-2)))
    betas = betas_dual_quenched_per_M[M]
mean_dual_entropies_quenched = torch.stack(mean_dual_entropies_quenched)
mean_dual_entropies_annealed = torch.stack(mean_dual_entropies_annealed)
all_Ms = torch.tensor(all_Ms)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ctf = ax.contourf(betas, all_Ms/K, mean_dual_entropies_quenched/torch.as_tensor(K).log(), levels=100, cmap='coolwarm', vmin=0.0, vmax=1.0)
plt.colorbar(ctf, label='$H(w)/log(K)$')
ax.set_yscale('log')
ax.set_xscale('log')
ax.set_xlabel('$\\beta$')
ax.set_ylabel('$M/K$')

ax.set_title('Quenched Entropies')
plt.tight_layout()
plt.savefig(f'paper_results/plots/onestep_weights_centered={is_centered}_quenched_N={N}_K={K}_rho0={rho0:.2f}_rho1={rho1:.2f}_entropies.pdf', bbox_inches='tight')
plt.show()


fig, ax = plt.subplots(figsize=(4, 3))
ctf = ax.contourf(betas, all_Ms/K, mean_dual_entropies_annealed/torch.as_tensor(K).log(), levels=100, cmap='coolwarm', vmin=0.0, vmax=1.0)
plt.colorbar(ctf, label='$H(w)/log(K)$')
ax.set_yscale('log')
ax.set_xscale('log')
ax.set_xlabel('$\\beta$')
ax.set_ylabel('$M/K$')
ax.set_title('Annealed Entropies')
plt.tight_layout()
plt.savefig(f'paper_results/plots/onestep_weights_centered={is_centered}_annealed_N={N}_K={K}_rho0={rho0:.2f}_rho1={rho1:.2f}_entropies.pdf', bbox_inches='tight')
plt.show()

In [ ]:
for M, probs in probs_dual_quenched_per_M.items():
    block_weights = probs.view(*probs.shape[:-1], M, K//M).sum(dim=-1)
    plt.plot(block_weights[:,0,:,:].var(dim=1).mean(dim=-1).sqrt(), label=f'M={M}')
plt.legend()
plt.xlabel('Beta')
plt.ylabel('Mean Std of Block Weights')
plt.title('Quenched Block Weight Variability')
plt.xscale('log')
plt.tight_layout()
plt.show()

In [129]:
from sklearn.decomposition import PCA
for M, probs in probs_dual_annealed_per_M.items():
    if M == 1:
        continue
    pattern_idx = 0
    selected_patterns = patterns_annealed_per_M[M][pattern_idx]
    selected_probs = probs[:,pattern_idx]
    selected_block_weights = selected_probs.view(*selected_probs.shape[:-1], M, K//M).sum(dim=-1)
    retrieved_pattern_indices = selected_probs[-1,:,:].argmax(dim=-1)
    retrieved_block_indices = selected_block_weights[-1,:,:].argmax(dim=-1)
    print(torch.unique(retrieved_pattern_indices, return_counts=True))
    print(torch.unique(retrieved_block_indices, return_counts=True))
    #sp_pca = PCA(n_components=2)
    #sp_pca.fit(selected_patterns)
    #patterns_pca = sp_pca.transform(selected_patterns)
    #average_pattern = torch.einsum("bck,ki->bci", selected_probs, selected_patterns)
    #average_pattern_pca = sp_pca.transform(average_pattern.view(-1, average_pattern.shape[-1]).numpy()).reshape(*average_pattern.shape[:-1], 2)
    #plt.scatter(patterns_pca[:,0], patterns_pca[:,1], c='gray', alpha=0.5, label='Patterns')
    #for ic_idx in range(average_pattern_pca.shape[1]):
    #    plt.plot(average_pattern_pca[:,ic_idx,0], average_pattern_pca[:,ic_idx,1], label=f'IC {ic_idx}')
    #plt.title(f'Quenched Dynamics: M={M}, Pattern {pattern_idx}')
    #plt.show()

(tensor([213, 250]), tensor([53, 43]))
(tensor([106, 125]), tensor([53, 43]))
(tensor([110, 226]), tensor([49, 47]))
(tensor([3, 7]), tensor([49, 47]))
(tensor([ 40, 231]), tensor([51, 45]))
(tensor([10, 57]), tensor([51, 45]))
(tensor([203, 223]), tensor([49, 47]))
(tensor([25, 27]), tensor([49, 47]))
(tensor([100, 168, 194, 205]), tensor([ 2,  1, 46, 47]))
(tensor([100, 168, 194, 205]), tensor([ 2,  1, 46, 47]))
(tensor([ 97, 217]), tensor([48, 48]))
(tensor([1, 3]), tensor([48, 48]))
(tensor([124, 240]), tensor([50, 46]))
(tensor([ 7, 15]), tensor([50, 46]))
(tensor([ 97, 138]), tensor([56, 40]))
(tensor([0, 1]), tensor([56, 40]))


In [ ]:
def stripplot_scores(
    scores,
    times=None,
    class_labels=None,
    ax=None,
    offset=0.3,
    jitter=0.0,
    seed=0,
    **scatter_kwargs,
):
    """
    Plot K class scores horizontally against time vertically.

    Parameters
    ----------
    scores : array, shape (T, K)
    times : array, shape (T,), optional
    class_labels : sequence of length K, optional
    ax : matplotlib axis, optional
    offset : float
        Total vertical separation between classes at each time.
    jitter : float
        Random vertical jitter amplitude.
    seed : int
    scatter_kwargs :
        Passed to ax.scatter.

    Returns
    -------
    fig, ax
    """
    scores = np.asarray(scores)
    if scores.ndim != 2:
        raise ValueError("scores must have shape (T, K)")

    T, K = scores.shape

    if times is None:
        times = np.arange(T)
    times = np.asarray(times)

    if times.shape != (T,):
        raise ValueError(f"times must have shape ({T},)")

    if class_labels is None:
        class_labels = [f"Class {k}" for k in range(K)]

    if len(class_labels) != K:
        raise ValueError(f"class_labels must have length {K}")

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 7))
    else:
        fig = ax.figure

    rng = np.random.default_rng(seed)
    offsets = np.linspace(-offset, offset, K)

    defaults = {"s": 25, "alpha": 0.8}
    defaults.update(scatter_kwargs)

    for k in range(K):
        noise = rng.uniform(-jitter, jitter, T) if jitter > 0 else 0

        ax.scatter(
            scores[:, k],
            times + offsets[k] + noise,
            label=class_labels[k],
            **defaults,
        )

    ax.set_xlabel("Score")
    ax.set_ylabel("Time")
    ax.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()

    return fig, ax